In [2]:
import pandas as pd

from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

print(Path.cwd())

from core.forecasting.dataset import load_daily_sales, create_daily_series
from core.forecasting.profiling import profile_demand

daily_sales = load_daily_sales()
forecasting_data = create_daily_series(daily_sales)

E:\Programming\Python\StockWise


In [2]:
# 1. Inspect Processed Data

print("Processed data inspection...")

print("\n--- Basic Information ---")
print(f"Rows: {len(daily_sales):,}")
print(f"Columns: {len(daily_sales.columns)}")
print(f"Columns: {list(daily_sales.columns)}")

unique_products = daily_sales["StockCode"].nunique()

print(f"\nUnique products: {unique_products:,}")

date_min = daily_sales["Date"].min()
date_max = daily_sales["Date"].max()

print(f"Date range: {date_min.date()} to {date_max.date()}")

product_date_combinations = (
    daily_sales.groupby(["StockCode", "Date"]).ngroups
)

print(
    f"Unique product-day combinations: "
    f"{product_date_combinations:,}"
)

print("\n--- Missing Values ---")

missing = daily_sales.isna().sum()

if missing.sum() == 0:
    print("No missing values.")
else:
    print(missing[missing > 0])

duplicates = daily_sales.duplicated(
    subset=["Date", "StockCode"]
).sum()

print("\n--- Duplicates ---")
print(f"Duplicate product-day rows: {duplicates:,}")

print("\n--- Demand Statistics ---")
print(daily_sales["Demand"].describe())

daily_demand = (
    daily_sales
    .groupby("Date")["Demand"]
    .sum()
)

print("\n--- Daily Demand ---")
print(daily_demand.describe())

product_demand = (
    daily_sales
    .groupby("StockCode")["Demand"]
    .sum()
    .sort_values(ascending=False)
)

print("\n--- Top 10 Products by Total Demand ---")
print(product_demand.head(10))

invalid_demand = daily_sales[
    daily_sales["Demand"] <= 0
]

print("\n--- Demand Validation ---")

if len(invalid_demand) == 0:
    print("All processed demand values are positive.")
else:
    print(
        f"Found {len(invalid_demand):,} "
        f"rows with demand <= 0."
    )

Q1 = daily_demand.quantile(0.25)
Q3 = daily_demand.quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

unusual = daily_demand[
    (daily_demand < lower)
    | (daily_demand > upper)
]

print("\n--- Unusual Daily Demand ---")
print(f"Number of unusual days: {len(unusual):,}")

if len(unusual) > 0:
    print("\nHighest unusual demand days:")
    print(
        unusual
        .sort_values(ascending=False)
        .head(10)
    )

Processed data inspection...

--- Basic Information ---
Rows: 276,843
Columns: 3
Columns: ['Date', 'StockCode', 'Demand']

Unique products: 3,941
Date range: 2010-12-01 to 2011-12-09
Unique product-day combinations: 276,843

--- Missing Values ---
No missing values.

--- Duplicates ---
Duplicate product-day rows: 0

--- Demand Statistics ---
count    276843.000000
mean         20.448344
std         219.089021
min           1.000000
25%           2.000000
50%           6.000000
75%          18.000000
max       80995.000000
Name: Demand, dtype: float64

--- Daily Demand ---
count      305.000000
mean     18560.593443
std      10710.818391
min       2048.000000
25%      11896.000000
50%      16349.000000
75%      22563.000000
max      93979.000000
Name: Demand, dtype: float64

--- Top 10 Products by Total Demand ---
StockCode
23843     80995
23166     78033
22197     56921
84077     55047
85099B    48478
85123A    41664
84879     36461
21212     36419
23084     31722
22492     26633
Name:

In [3]:
# 2. Inspect Demand

print("Demand investigation...")

print("\n--- Highest Product-Day Demand ---")

top_product_days = (
    daily_sales
    .sort_values("Demand", ascending=False)
    .head(20)
)

print(
    top_product_days.to_string(index=False)
)

daily_demand = (
    daily_sales
    .groupby("Date")["Demand"]
    .sum()
    .sort_values(ascending=False)
)

print("\n--- Highest-Demand Days ---")
print(daily_demand.head(20))

top_product = daily_sales.loc[
    daily_sales["Demand"].idxmax(),
    "StockCode"
]

print(f"\n--- Product {top_product} ---")

product_history = (
    daily_sales[
        daily_sales["StockCode"] == top_product
    ]
    .sort_values("Date")
)

print(
    product_history.to_string(index=False)
)

print("\n--- Final 10 Dates ---")

final_dates = (
    daily_sales
    .groupby("Date")["Demand"]
    .sum()
    .tail(10)
)

print(final_dates)

daily_products = (
    daily_sales
    .groupby("Date")["StockCode"]
    .nunique()
)

print("\n--- Products Sold Per Day ---")
print(daily_products.describe())

print("\nDays with highest number of products:")

print(
    daily_products
    .sort_values(ascending=False)
    .head(10)
)

monthly_demand = (
    daily_sales
    .set_index("Date")
    .resample("ME")["Demand"]
    .sum()
)

print("\n--- Monthly Demand ---")
print(monthly_demand)

Demand investigation...

--- Highest Product-Day Demand ---
      Date StockCode  Demand
2011-12-09     23843   80995
2011-01-18     23166   74215
2011-11-25     84826   12540
2011-01-28     37413    5568
2011-10-27     84077    4848
2011-05-27     22197    4314
2011-06-09    85123A    4015
2011-02-22     22053    3906
2011-08-04     84879    3359
2010-12-02     84077    3264
2011-07-19     18007    3206
2011-01-11     21108    3132
2011-04-18     21108    3123
2011-01-11    85123A    3114
2011-12-08     22197    3111
2011-07-14     23343    3100
2011-11-02     16014    3020
2011-03-09     84988    3012
2011-07-31     84568    2880
2011-10-07     20971    2868

--- Highest-Demand Days ---
Date
2011-12-09    93979
2011-01-18    82978
2011-11-14    47631
2011-10-05    46338
2011-12-05    44664
2011-09-20    43733
2011-12-07    41980
2011-08-04    41148
2011-10-20    40919
2011-08-11    40299
2011-11-23    39420
2011-11-10    39239
2011-11-09    38269
2011-05-12    37833
2011-09-13    369

In [4]:
# 3. Inspect and Validate Forecasting Dataset

product_history = (
    forecasting_data
    .groupby("StockCode")["Date"]
    .agg(
        FirstDate="min",
        LastDate="max",
        Days="nunique",
    )
)

raw_product_history = (
    daily_sales
    .groupby("StockCode")["Date"]
    .agg(
        FirstSaleDate="min",
        LastSaleDate="max",
    )
)

history_validation = raw_product_history.merge(
    product_history,
    left_index=True,
    right_index=True,
)

history_validation["FirstDateCorrect"] = (
    history_validation["FirstSaleDate"]
    == history_validation["FirstDate"]
)

history_validation["LastDateCorrect"] = (
    history_validation["LastSaleDate"]
    == history_validation["LastDate"]
)

print("\n--- Product History ---")
print(product_history["Days"].describe())

print("\nProducts with fewer than 28 days:")
print((product_history["Days"] < 28).sum())

print("\nProducts with fewer than 56 days:")
print((product_history["Days"] < 56).sum())

print("\n--- Active-Date Boundary Validation ---")

print(
    "Incorrect first dates:",
    (~history_validation["FirstDateCorrect"]).sum(),
)

print(
    "Incorrect last dates:",
    (~history_validation["LastDateCorrect"]).sum(),
)

if (
    history_validation["FirstDateCorrect"].all()
    and history_validation["LastDateCorrect"].all()
):
    print(
        "All products start and end on their observed "
        "first and last sales dates."
    )
else:
    print("Active-date boundary validation failed.")

print("\n--- Forecasting Dataset ---")

print(f"Rows: {len(forecasting_data):,}")
print(
    f"Products: "
    f"{forecasting_data['StockCode'].nunique():,}"
)

print(
    f"Date range: "
    f"{forecasting_data['Date'].min().date()} "
    f"to "
    f"{forecasting_data['Date'].max().date()}"
)

print("\n--- Demand ---")
print(forecasting_data["Demand"].describe())

zero_demand_rows = (
    forecasting_data["Demand"] == 0
).sum()

positive_demand_rows = (
    forecasting_data["Demand"] > 0
).sum()

print("\n--- Demand Accounting ---")
print(
    f"Positive-demand rows: "
    f"{positive_demand_rows:,}"
)
print(
    f"Zero-demand rows: "
    f"{zero_demand_rows:,}"
)
print(
    f"Total rows: "
    f"{len(forecasting_data):,}"
)

print("\n--- Missing Values ---")
print(forecasting_data.isna().sum())


--- Product History ---
count    3941.000000
mean      273.502664
std       126.426580
min         1.000000
25%       180.000000
50%       353.000000
75%       372.000000
max       374.000000
Name: Days, dtype: float64

Products with fewer than 28 days:
274

Products with fewer than 56 days:
400

--- Active-Date Boundary Validation ---
Incorrect first dates: 0
Incorrect last dates: 0
All products start and end on their observed first and last sales dates.

--- Forecasting Dataset ---
Rows: 1,077,874
Products: 3,941
Date range: 2010-12-01 to 2011-12-09

--- Demand ---
count    1.077874e+06
mean     5.251988e+00
std      1.113920e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      8.099500e+04
Name: Demand, dtype: float64

--- Demand Accounting ---
Positive-demand rows: 276,843
Zero-demand rows: 801,031
Total rows: 1,077,874

--- Missing Values ---
Date          0
StockCode     0
Demand        0
ActiveDays    0
dtype: int64


In [5]:
# Validate that zero-demand rows exist only inside
# each product's observed active period.

zero_demand_outside_history = forecasting_data.merge(
    raw_product_history,
    left_on="StockCode",
    right_index=True,
    how="left",
)

outside_history = zero_demand_outside_history[
    (
        zero_demand_outside_history["Demand"] == 0
    )
    & (
        (
            zero_demand_outside_history["Date"]
            < zero_demand_outside_history["FirstSaleDate"]
        )
        |
        (
            zero_demand_outside_history["Date"]
            > zero_demand_outside_history["LastSaleDate"]
        )
    )
]

print("\n--- Zero-Demand Boundary Validation ---")
print(
    f"Zero-demand rows outside active history: "
    f"{len(outside_history):,}"
)

if outside_history.empty:
    print(
        "Validation passed: zero-demand rows exist only "
        "between each product's first and last observed sale."
    )
else:
    print("Validation failed.")


--- Zero-Demand Boundary Validation ---
Zero-demand rows outside active history: 0
Validation passed: zero-demand rows exist only between each product's first and last observed sale.


In [6]:
# 4. Profile Demand

demand_profile = profile_demand(forecasting_data)

print("\n--- Demand Profile ---")
print(f"Products: {len(demand_profile):,}")
print(
    f"Rows in forecasting dataset: "
    f"{len(forecasting_data):,}"
)

print("\n--- Overall Demand ---")

print(
    demand_profile[
        [
            "ActiveDays",
            "TotalDemand",
            "MeanDemand",
            "MedianDemand",
            "StdDemand",
            "MinDemand",
            "MaxDemand",
            "ZeroDemandPct",
            "PositiveDemandPct",
            "CoefficientOfVariation",
        ]
    ]
    .describe()
)

print("\n--- Products With Highest Zero-Demand Percentage ---")

print(
    demand_profile[
        [
            "StockCode",
            "ActiveDays",
            "PositiveDemandDays",
            "ZeroDemandPct",
            "MeanDemand",
            "TotalDemand",
        ]
    ]
    .sort_values(
        "ZeroDemandPct",
        ascending=False,
    )
    .head(10)
    .to_string(index=False)
)

print("\n--- Products With Highest Coefficient of Variation ---")

print(
    demand_profile[
        [
            "StockCode",
            "ActiveDays",
            "MeanDemand",
            "StdDemand",
            "CoefficientOfVariation",
            "MaxDemand",
        ]
    ]
    .sort_values(
        "CoefficientOfVariation",
        ascending=False,
    )
    .head(10)
    .to_string(index=False)
)

print("\n--- Products With Highest Total Demand ---")

print(
    demand_profile[
        [
            "StockCode",
            "ActiveDays",
            "TotalDemand",
            "MeanDemand",
            "MedianDemand",
            "MaxDemand",
        ]
    ]
    .sort_values(
        "TotalDemand",
        ascending=False,
    )
    .head(10)
    .to_string(index=False)
)


--- Demand Profile ---
Products: 3,941
Rows in forecasting dataset: 1,077,874

--- Overall Demand ---
        ActiveDays   TotalDemand    MeanDemand  MedianDemand    StdDemand  \
count  3941.000000   3941.000000   3941.000000   3941.000000  3781.000000   
mean    273.502664   1436.432631     26.242475     22.368942    15.890075   
std     126.426580   3604.642278   1290.318606   1290.343233    72.584152   
min       1.000000      1.000000      0.006803      0.000000     0.000000   
25%     180.000000     57.000000      0.387879      0.000000     1.597135   
50%     353.000000    378.000000      1.670330      0.000000     5.899669   
75%     372.000000   1398.000000      5.323529      0.000000    17.222915   
max     374.000000  80995.000000  80995.000000  80995.000000  4109.868865   

          MinDemand     MaxDemand  ZeroDemandPct  PositiveDemandPct  \
count   3941.000000   3941.000000    3941.000000        3941.000000   
mean      21.121543    192.137529      70.470832          29.

In [7]:
# 5. Calculate demand-pattern metrics for each product.
#
# ADI (Average Demand Interval):
#     Active days / positive-demand days
#
# CV² (Squared Coefficient of Variation):
#     (standard deviation of positive-demand size
#      / mean positive-demand size)²
#
# Demand patterns:
#     Smooth       -> ADI < 1.32 and CV² < 0.49
#     Erratic      -> ADI < 1.32 and CV² >= 0.49
#     Intermittent -> ADI >= 1.32 and CV² < 0.49
#     Lumpy        -> ADI >= 1.32 and CV² >= 0.49
#     Unknown      -> fewer than two positive-demand observations

In [8]:
positive_demand = forecasting_data[
    forecasting_data["Demand"] > 0
]

positive_stats = (
    positive_demand
    .groupby("StockCode")["Demand"]
    .agg(
        MeanPositiveDemand="mean",
        StdPositiveDemand="std",
        PositiveDemandDays="count",
    )
    .reset_index()
)

activity_stats = (
    forecasting_data
    .groupby("StockCode")["Demand"]
    .agg(
        ActiveDays="size",
        TotalDemand="sum",
    )
    .reset_index()
)

demand_pattern = activity_stats.merge(
    positive_stats,
    on="StockCode",
    how="left",
)

demand_pattern["ADI"] = (
    demand_pattern["ActiveDays"]
    / demand_pattern["PositiveDemandDays"]
)

demand_pattern["CV2"] = (
    demand_pattern["StdPositiveDemand"]
    / demand_pattern["MeanPositiveDemand"]
) ** 2

demand_pattern["ZeroDemandDays"] = (
    demand_pattern["ActiveDays"]
    - demand_pattern["PositiveDemandDays"]
)

demand_pattern["ZeroDemandPct"] = (
    demand_pattern["ZeroDemandDays"]
    / demand_pattern["ActiveDays"]
    * 100
)

demand_pattern["PositiveDemandPct"] = (
    demand_pattern["PositiveDemandDays"]
    / demand_pattern["ActiveDays"]
    * 100
)

demand_pattern["DemandPattern"] = "Unknown"

demand_pattern.loc[
    (demand_pattern["ADI"] < 1.32)
    & (demand_pattern["CV2"] < 0.49),
    "DemandPattern",
] = "Smooth"

demand_pattern.loc[
    (demand_pattern["ADI"] < 1.32)
    & (demand_pattern["CV2"] >= 0.49),
    "DemandPattern",
] = "Erratic"

demand_pattern.loc[
    (demand_pattern["ADI"] >= 1.32)
    & (demand_pattern["CV2"] < 0.49),
    "DemandPattern",
] = "Intermittent"

demand_pattern.loc[
    (demand_pattern["ADI"] >= 1.32)
    & (demand_pattern["CV2"] >= 0.49),
    "DemandPattern",
] = "Lumpy"

demand_pattern.loc[
    demand_pattern["PositiveDemandDays"] < 2,
    "DemandPattern",
] = "Unknown"

demand_pattern = demand_pattern[
    [
        "StockCode",
        "ActiveDays",
        "PositiveDemandDays",
        "ZeroDemandDays",
        "PositiveDemandPct",
        "ZeroDemandPct",
        "TotalDemand",
        "MeanPositiveDemand",
        "StdPositiveDemand",
        "ADI",
        "CV2",
        "DemandPattern",
    ]
]

In [9]:
print("=" * 70)
print("DEMAND PATTERN CLASSIFICATION")
print("=" * 70)

print(
    f"Products analyzed: "
    f"{len(demand_pattern):,}"
)

print(
    f"Forecasting rows: "
    f"{len(forecasting_data):,}"
)

pattern_counts = (
    demand_pattern["DemandPattern"]
    .value_counts()
    .reindex(
        [
            "Smooth",
            "Erratic",
            "Intermittent",
            "Lumpy",
            "Unknown",
        ],
        fill_value=0,
    )
)

pattern_percentages = (
    pattern_counts
    / len(demand_pattern)
    * 100
)

pattern_summary = pd.DataFrame(
    {
        "Products": pattern_counts,
        "Percentage": pattern_percentages,
    }
)

print("\n--- Demand Pattern Distribution ---")
print(pattern_summary.round(2))

print("\n--- ADI Statistics ---")
print(
    demand_pattern["ADI"]
    .describe()
    .round(2)
)

print("\n--- CV² Statistics ---")
print(
    demand_pattern["CV2"]
    .describe()
    .round(2)
)

print("\n--- Demand Frequency Statistics ---")

print(
    demand_pattern[
        [
            "ActiveDays",
            "PositiveDemandDays",
            "ZeroDemandDays",
            "PositiveDemandPct",
            "ZeroDemandPct",
        ]
    ]
    .describe()
    .round(2)
)

DEMAND PATTERN CLASSIFICATION
Products analyzed: 3,941
Forecasting rows: 1,077,874

--- Demand Pattern Distribution ---
               Products  Percentage
DemandPattern                      
Smooth                7        0.18
Erratic             110        2.79
Intermittent        578       14.67
Lumpy              3086       78.30
Unknown             160        4.06

--- ADI Statistics ---
count    3941.00
mean       10.93
std        16.71
min         1.00
25%         2.29
50%         4.63
75%        11.71
max       179.50
Name: ADI, dtype: float64

--- CV² Statistics ---
count    3781.00
mean        2.55
std         4.19
min         0.00
25%         0.77
50%         1.46
75%         2.76
max       112.98
Name: CV2, dtype: float64

--- Demand Frequency Statistics ---
       ActiveDays  PositiveDemandDays  ZeroDemandDays  PositiveDemandPct  \
count     3941.00             3941.00         3941.00            3941.00   
mean       273.50               70.25          203.26              

In [10]:
print("=" * 70)
print("PRODUCTS WITH HIGHEST ADI")
print("=" * 70)

print(
    demand_pattern[
        [
            "StockCode",
            "ActiveDays",
            "PositiveDemandDays",
            "ADI",
            "CV2",
            "ZeroDemandPct",
            "DemandPattern",
        ]
    ]
    .sort_values("ADI", ascending=False)
    .head(20)
    .to_string(index=False)
)

PRODUCTS WITH HIGHEST ADI
StockCode  ActiveDays  PositiveDemandDays        ADI      CV2  ZeroDemandPct DemandPattern
   79151B         359                   2 179.500000 1.338843      99.442897         Lumpy
   90128C         294                   2 147.000000 0.000000      99.319728  Intermittent
   84660b         273                   2 136.500000 0.000000      99.267399  Intermittent
    90140         268                   2 134.000000 0.000000      99.253731  Intermittent
    45013         257                   2 128.500000 0.222222      99.221790  Intermittent
   90176A         249                   2 124.500000 0.222222      99.196787  Intermittent
    44228         247                   2 123.500000 0.222222      99.190283  Intermittent
   35817P         241                   2 120.500000 0.222222      99.170124  Intermittent
    90074         358                   3 119.333333 0.187500      99.162011  Intermittent
   90128A         230                   2 115.000000 0.222222   

In [11]:
print("=" * 70)
print("PRODUCTS WITH HIGHEST CV²")
print("=" * 70)

print(
    demand_pattern[
        [
            "StockCode",
            "ActiveDays",
            "PositiveDemandDays",
            "ADI",
            "CV2",
            "ZeroDemandPct",
            "DemandPattern",
        ]
    ]
    .sort_values("CV2", ascending=False)
    .head(20)
    .to_string(index=False)
)

PRODUCTS WITH HIGHEST CV²
StockCode  ActiveDays  PositiveDemandDays      ADI        CV2  ZeroDemandPct DemandPattern
    23166         326                 125 2.608000 112.979168      61.656442         Lumpy
      DOT         374                 227 1.647577  77.791994      39.304813         Lumpy
    37413         312                  60 5.200000  47.132667      80.769231         Lumpy
    22476         374                  94 3.978723  43.189441      74.866310         Lumpy
    84988         373                 241 1.547718  41.408469      35.388740         Lumpy
    22508         373                 186 2.005376  40.014867      50.134048         Lumpy
    22350         373                  98 3.806122  39.494938      73.726542         Lumpy
    22413         373                 234 1.594017  33.927093      37.265416         Lumpy
    48111         374                 157 2.382166  30.540351      58.021390         Lumpy
    21623         372                 152 2.447368  30.337601   

In [12]:
print("=" * 70)
print("PRODUCTS WITH HIGHEST TOTAL DEMAND")
print("=" * 70)

print(
    demand_pattern[
        [
            "StockCode",
            "TotalDemand",
            "ActiveDays",
            "PositiveDemandDays",
            "ADI",
            "CV2",
            "DemandPattern",
        ]
    ]
    .sort_values(
        "TotalDemand",
        ascending=False,
    )
    .head(20)
    .to_string(index=False)
)

PRODUCTS WITH HIGHEST TOTAL DEMAND
StockCode  TotalDemand  ActiveDays  PositiveDemandDays      ADI        CV2 DemandPattern
    23843      80995.0           1                   1 1.000000        NaN       Unknown
    23166      78033.0         326                 125 2.608000 112.979168         Lumpy
    22197      56921.0         374                 294 1.272109   4.589037       Erratic
    84077      55047.0         373                 235 1.587234   4.177576         Lumpy
   85099B      48478.0         374                 301 1.242525   1.383441       Erratic
   85123A      41664.0         374                 305 1.226230   5.798393       Erratic
    84879      36461.0         374                 300 1.246667   3.509701       Erratic
    21212      36419.0         374                 298 1.255034   2.298675       Erratic
    23084      31722.0         214                 140 1.528571   3.567518         Lumpy
    22492      26633.0         373                 208 1.793269   2.522978 

In [13]:
print("=" * 70)
print("PRODUCTS WITH HIGHEST ZERO-DEMAND PERCENTAGE")
print("=" * 70)

print(
    demand_pattern[
        [
            "StockCode",
            "ActiveDays",
            "PositiveDemandDays",
            "ZeroDemandDays",
            "ZeroDemandPct",
            "ADI",
            "CV2",
            "DemandPattern",
        ]
    ]
    .sort_values(
        "ZeroDemandPct",
        ascending=False,
    )
    .head(20)
    .to_string(index=False)
)

PRODUCTS WITH HIGHEST ZERO-DEMAND PERCENTAGE
StockCode  ActiveDays  PositiveDemandDays  ZeroDemandDays  ZeroDemandPct        ADI      CV2 DemandPattern
   79151B         359                   2             357      99.442897 179.500000 1.338843         Lumpy
   90128C         294                   2             292      99.319728 147.000000 0.000000  Intermittent
   84660b         273                   2             271      99.267399 136.500000 0.000000  Intermittent
    90140         268                   2             266      99.253731 134.000000 0.000000  Intermittent
    45013         257                   2             255      99.221790 128.500000 0.222222  Intermittent
   90176A         249                   2             247      99.196787 124.500000 0.222222  Intermittent
    44228         247                   2             245      99.190283 123.500000 0.222222  Intermittent
   35817P         241                   2             239      99.170124 120.500000 0.222222  Inter

In [14]:
unknown_products = demand_pattern[
    demand_pattern["DemandPattern"] == "Unknown"
].copy()

print("=" * 70)
print("UNKNOWN / INSUFFICIENT-DATA PRODUCTS")
print("=" * 70)

print(
    f"Products classified as Unknown: "
    f"{len(unknown_products):,}"
)

if len(unknown_products) > 0:

    print(
        unknown_products[
            [
                "StockCode",
                "ActiveDays",
                "PositiveDemandDays",
                "ZeroDemandDays",
                "ADI",
                "CV2",
                "DemandPattern",
            ]
        ]
        .sort_values(
            ["PositiveDemandDays", "ActiveDays"],
            ascending=[True, False],
        )
        .head(20)
        .to_string(index=False)
    )

UNKNOWN / INSUFFICIENT-DATA PRODUCTS
Products classified as Unknown: 160
StockCode  ActiveDays  PositiveDemandDays  ZeroDemandDays  ADI  CV2 DemandPattern
   16161M           1                   1               0  1.0  NaN       Unknown
   16169N           1                   1               0  1.0  NaN       Unknown
    17001           1                   1               0  1.0  NaN       Unknown
   17028J           1                   1               0  1.0  NaN       Unknown
    20678           1                   1               0  1.0  NaN       Unknown
    20703           1                   1               0  1.0  NaN       Unknown
    20816           1                   1               0  1.0  NaN       Unknown
    20849           1                   1               0  1.0  NaN       Unknown
    20860           1                   1               0  1.0  NaN       Unknown
    20871           1                   1               0  1.0  NaN       Unknown
    20950           1    

In [15]:
print("=" * 70)
print("CLASSIFICATION VALIDATION")
print("=" * 70)

print(
    "Missing ADI values:",
    demand_pattern["ADI"].isna().sum(),
)

print(
    "Missing CV² values:",
    demand_pattern["CV2"].isna().sum(),
)

print(
    "Missing demand patterns:",
    demand_pattern["DemandPattern"].isna().sum(),
)

classified = demand_pattern[
    demand_pattern["DemandPattern"] != "Unknown"
].copy()

classification_errors = classified[
    (
        (classified["DemandPattern"] == "Smooth")
        & ~(
            (classified["ADI"] < 1.32)
            & (classified["CV2"] < 0.49)
        )
    )
    |
    (
        (classified["DemandPattern"] == "Erratic")
        & ~(
            (classified["ADI"] < 1.32)
            & (classified["CV2"] >= 0.49)
        )
    )
    |
    (
        (classified["DemandPattern"] == "Intermittent")
        & ~(
            (classified["ADI"] >= 1.32)
            & (classified["CV2"] < 0.49)
        )
    )
    |
    (
        (classified["DemandPattern"] == "Lumpy")
        & ~(
            (classified["ADI"] >= 1.32)
            & (classified["CV2"] >= 0.49)
        )
    )
]

print(
    "Classification errors:",
    len(classification_errors),
)

print("\n--- Final Demand Pattern Table ---")

print(
    demand_pattern
    .head(20)
    .to_string(index=False)
)

CLASSIFICATION VALIDATION
Missing ADI values: 0
Missing CV² values: 160
Missing demand patterns: 0
Classification errors: 0

--- Final Demand Pattern Table ---
StockCode  ActiveDays  PositiveDemandDays  ZeroDemandDays  PositiveDemandPct  ZeroDemandPct  TotalDemand  MeanPositiveDemand  StdPositiveDemand       ADI      CV2 DemandPattern
    10002         139                  47              92          33.812950      66.187050       1040.0           22.127660          36.845516  2.957447 2.772671         Lumpy
    10080         268                  22             246           8.208955      91.791045        495.0           22.500000          37.249161 12.181818 2.740741         Lumpy
    10120         367                  24             343           6.539510      93.460490        193.0            8.041667          10.178747 15.291667 1.602125         Lumpy
   10123C         119                   3             116           2.521008      97.478992          5.0            1.666667        

# Additional data validation:
1. Validate first/last dates
2. Validate zero-demand boundaries
3. Validate missing values
4. Validate demand totals
5. Inspect extreme observations
6. Create an exceptional-demand analysis dataset
7. Validate deterministic cleaning / pipeline consistency

## 1. Validate Product History Boundaries

The continuous daily series should begin on each product's first observed sale
date and end on its last observed sale date.

This validation checks that the series expansion does not create observations
before the first sale or after the last sale.

In [17]:
raw_boundaries = (
    daily_sales.groupby("StockCode")["Date"]
    .agg(RawFirstDate="min", RawLastDate="max")
    .reset_index()
)

series_boundaries = (
    forecasting_data.groupby("StockCode")["Date"]
    .agg(SeriesFirstDate="min", SeriesLastDate="max")
    .reset_index()
)

boundary_check = raw_boundaries.merge(series_boundaries, on="StockCode", how="outer")

boundary_check["FirstDateMatch"] = (
    boundary_check["RawFirstDate"] == boundary_check["SeriesFirstDate"]
)

boundary_check["LastDateMatch"] = (
    boundary_check["RawLastDate"] == boundary_check["SeriesLastDate"]
)

boundary_check["BoundaryMatch"] = (
    boundary_check["FirstDateMatch"] & boundary_check["LastDateMatch"]
)

boundary_check.head()

,StockCode,RawFirstDate,RawLastDate,SeriesFirstDate,SeriesLastDate,FirstDateMatch,LastDateMatch,BoundaryMatch
0,10002,2010-12-01,2011-04-18,2010-12-01,2011-04-18,True,True,True
1,10080,2011-02-27,2011-11-21,2011-02-27,2011-11-21,True,True,True
2,10120,2010-12-03,2011-12-04,2010-12-03,2011-12-04,True,True,True
3,10123C,2010-12-03,2011-03-31,2010-12-03,2011-03-31,True,True,True
4,10124A,2010-12-06,2011-11-06,2010-12-06,2011-11-06,True,True,True


In [18]:
boundary_check["BoundaryMatch"].value_counts()

BoundaryMatch
True    3941
Name: count, dtype: int64

In [19]:
boundary_check[~boundary_check["BoundaryMatch"]]

,StockCode,RawFirstDate,RawLastDate,SeriesFirstDate,SeriesLastDate,FirstDateMatch,LastDateMatch,BoundaryMatch


## 2. Validate Zero-Demand Boundaries

Zero-demand rows are valid only inside a product's active history.

No zero-demand observation should exist before a product's first observed sale
or after its last observed sale.

In [20]:
history = (
    daily_sales.groupby("StockCode")["Date"]
    .agg(FirstSaleDate="min", LastSaleDate="max")
    .reset_index()
)

zero_demand = forecasting_data[forecasting_data["Demand"] == 0].merge(
    history,
    on="StockCode",
    how="left"
)

zero_demand["OutsideActiveHistory"] = (
    (zero_demand["Date"] < zero_demand["FirstSaleDate"])
    | (zero_demand["Date"] > zero_demand["LastSaleDate"])
)

zero_demand["OutsideActiveHistory"].value_counts()

OutsideActiveHistory
False    801031
Name: count, dtype: int64

## 3. Validate Missing Values

The forecasting dataset should contain complete values for all fields required
by the forecasting pipeline.

In [21]:
forecasting_data.isna().sum()

Date          0
StockCode     0
Demand        0
ActiveDays    0
dtype: int64

In [22]:
missing_values = forecasting_data.isna().sum()

missing_check = pd.DataFrame({
    "MissingValues": missing_values,
    "Valid": missing_values == 0
})

missing_check

,MissingValues,Valid
Date,0,True
StockCode,0,True
Demand,0,True
ActiveDays,0,True


In [23]:
missing_check["Valid"].all()

np.True_

## 4. Validate Demand Totals

The daily demand aggregation should preserve the total quantity of all SALE
transactions.

The continuous-series expansion should also preserve the same total because
it only adds zero-demand product-days.

In [6]:
RAW_FILE = "data/raw/Online Retail.xlsx"

raw_data = pd.read_excel(RAW_FILE)

raw_sales_total = raw_data.loc[
    raw_data["Quantity"] > 0,
    "Quantity"
].sum()

daily_sales_total = daily_sales["Demand"].sum()

forecasting_total = forecasting_data["Demand"].sum()

demand_totals = pd.DataFrame({
    "Dataset": [
        "Raw positive sales",
        "Daily sales",
        "Forecasting series"
    ],
    "TotalDemand": [
        raw_sales_total,
        daily_sales_total,
        forecasting_total
    ]
})

demand_totals

,Dataset,TotalDemand
0,Raw positive sales,5660981.0
1,Daily sales,5660981.0
2,Forecasting series,5660981.0


In [26]:
demand_totals["TotalDemand"].nunique()

1

In [27]:
forecasting_total == raw_sales_total

np.True_

## 5. Inspect Extreme Demand Observations

Extreme demand observations are investigated separately because a small number
of unusually large transactions can have a disproportionate effect on
forecasting models.

These observations are not removed automatically. They are retained for
traceability and evaluated as potential exceptional-demand events.

In [28]:
extreme_demand = (
    forecasting_data[forecasting_data["Demand"] > 0]
    .nlargest(20, "Demand")
    [["Date", "StockCode", "Demand"]]
)

extreme_demand

,Date,StockCode,Demand
701767,2011-12-09,23843,80995.0
650016,2011-01-18,23166,74215.0
871801,2011-11-25,84826,12540.0
723030,2011-01-28,37413,5568.0
815741,2011-10-27,84077,4848.0
351579,2011-05-27,22197,4314.0
938920,2011-06-09,85123A,4015.0
304847,2011-02-22,22053,3906.0
880405,2011-08-04,84879,3359.0
815412,2010-12-02,84077,3264.0


In [30]:
exceptional_data = forecasting_data.copy()

product_frequency = (
    forecasting_data.groupby("StockCode")
    .size()
    .rename("ProductHistoryDays")
)

positive_frequency = (
    forecasting_data[forecasting_data["Demand"] > 0]
    .groupby("StockCode")
    .size()
    .rename("PositiveDemandDays")
)

exceptional_data = exceptional_data.merge(product_frequency, on="StockCode")
exceptional_data = exceptional_data.merge(positive_frequency, on="StockCode")

quantity_threshold = forecasting_data["Demand"].quantile(0.999)

exceptional_data["ExtremeQuantity"] = (
    exceptional_data["Demand"] >= quantity_threshold
)

exceptional_data["RareProduct"] = (
    exceptional_data["PositiveDemandDays"] == 1
)

exceptional_data["ExceptionalDemand"] = (
    exceptional_data["ExtremeQuantity"]
    & exceptional_data["RareProduct"]
)

exceptional_data[
    exceptional_data["ExceptionalDemand"]
][
    [
        "Date",
        "StockCode",
        "Demand",
        "ProductHistoryDays",
        "PositiveDemandDays",
        "ExtremeQuantity",
        "RareProduct",
        "ExceptionalDemand",
    ]
].sort_values("Demand", ascending=False)

,Date,StockCode,Demand,ProductHistoryDays,PositiveDemandDays,ExtremeQuantity,RareProduct,ExceptionalDemand
701767,2011-12-09,23843,80995.0,1,1,True,True,True
848025,2011-02-04,84611B,1287.0,1,1,True,True,True


## 6. Investigate Exceptional Demand and Cancellations

Some exceptional demand observations may be followed by cancellation records.

The source data does not provide an explicit relationship between a
cancellation and its original sale, so this analysis is descriptive only.
It is not used to alter the production demand dataset.

In [31]:
cancellations = raw_data[
    raw_data["InvoiceNo"].astype(str).str.startswith("C")
].copy()

cancellations["InvoiceNo"] = cancellations["InvoiceNo"].astype(str).str.strip()

cancellations[
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice"]
].head()

,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice
141,C536379,D,-1,2010-12-01 09:41:00,27.50
154,C536383,35004C,-1,2010-12-01 09:49:00,4.65
235,C536391,22556,-12,2010-12-01 10:24:00,1.65
236,C536391,21984,-24,2010-12-01 10:24:00,0.29
237,C536391,21983,-24,2010-12-01 10:24:00,0.29


In [32]:
exceptional_products = (
    exceptional_data.loc[
        exceptional_data["ExceptionalDemand"],
        "StockCode"
    ]
    .unique()
)

cancellations[
    cancellations["StockCode"].isin(exceptional_products)
][
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice"]
].sort_values("Quantity")

,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice


In [33]:
exceptional_products

<StringArray>
['23843', '84611B']
Length: 2, dtype: str

In [34]:
raw_data[
    raw_data["StockCode"].isin(exceptional_products)
][
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice", "Description"]
].sort_values(["StockCode", "InvoiceDate"])

,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice,Description
82794,543257,84611B,-1430,2011-02-04 16:06:00,0.0,thrown away
82795,543258,84611B,1287,2011-02-04 16:06:00,0.0,NaN


In [35]:
exceptional_product_check = pd.DataFrame({
    "StockCode": exceptional_products
})

exceptional_product_check["RawTransactions"] = (
    exceptional_product_check["StockCode"]
    .map(raw_data["StockCode"].value_counts())
    .fillna(0)
    .astype(int)
)

exceptional_product_check

,StockCode,RawTransactions
0,23843,0
1,84611B,2


In [36]:
exceptional_product_check["RawTransactions"].eq(0).sum()

np.int64(1)

In [37]:
raw_data.loc[
    raw_data["StockCode"].astype(str).isin(exceptional_products.astype(str)),
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice", "Description"]
].sort_values(["StockCode", "InvoiceDate"])

,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice,Description
540421,581483,23843,80995,2011-12-09 09:15:00,2.08,"PAPER CRAFT , LITTLE BIRDIE"
540422,C581484,23843,-80995,2011-12-09 09:27:00,2.08,"PAPER CRAFT , LITTLE BIRDIE"
82794,543257,84611B,-1430,2011-02-04 16:06:00,0.00,thrown away
82795,543258,84611B,1287,2011-02-04 16:06:00,0.00,NaN


In [38]:
exceptional_raw = raw_data[
    raw_data["StockCode"].astype(str).isin(
        exceptional_products.astype(str)
    )
].copy()

exceptional_raw[
    [
        "InvoiceNo",
        "StockCode",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "Description"
    ]
].sort_values(["StockCode", "InvoiceDate"])

,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice,Description
540421,581483,23843,80995,2011-12-09 09:15:00,2.08,"PAPER CRAFT , LITTLE BIRDIE"
540422,C581484,23843,-80995,2011-12-09 09:27:00,2.08,"PAPER CRAFT , LITTLE BIRDIE"
82794,543257,84611B,-1430,2011-02-04 16:06:00,0.00,thrown away
82795,543258,84611B,1287,2011-02-04 16:06:00,0.00,NaN


In [39]:
exceptional_raw.groupby("StockCode").agg(
    Transactions=("InvoiceNo", "count"),
    PositiveQuantity=("Quantity", lambda x: x[x > 0].sum()),
    NegativeQuantity=("Quantity", lambda x: x[x < 0].sum()),
    MaxQuantity=("Quantity", "max"),
    MinQuantity=("Quantity", "min")
).reset_index()

,StockCode,Transactions,PositiveQuantity,NegativeQuantity,MaxQuantity,MinQuantity
0,23843,2,80995,-80995,80995,-80995
1,84611B,2,1287,-1430,1287,-1430


In [40]:
exceptional_summary = exceptional_raw[
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice", "Description"]
].sort_values(["StockCode", "InvoiceDate"])

exceptional_summary

,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice,Description
540421,581483,23843,80995,2011-12-09 09:15:00,2.08,"PAPER CRAFT , LITTLE BIRDIE"
540422,C581484,23843,-80995,2011-12-09 09:27:00,2.08,"PAPER CRAFT , LITTLE BIRDIE"
82794,543257,84611B,-1430,2011-02-04 16:06:00,0.00,thrown away
82795,543258,84611B,1287,2011-02-04 16:06:00,0.00,NaN


### Exceptional-demand investigation

Two products were identified as exceptional demand cases.

Product `23843` recorded an unusually large positive quantity of 80,995 units, followed 12 minutes later by a cancellation of exactly the same quantity at the same unit price. This indicates that the transaction should not be treated as representative ordinary demand.

Product `84611B` recorded a positive quantity of 1,287 units and a negative quantity of 1,430 units at the same timestamp. The negative transaction is explicitly described as `thrown away`, indicating an operational inventory event rather than ordinary customer demand.

These observations are retained for analysis but are not removed automatically from the raw data. The production cleaning pipeline does not attempt to infer cancellation-to-sale relationships because the source dataset does not provide an explicit relationship between transactions.

Exceptional events will be handled separately from ordinary demand forecasting.

In [11]:
zero_price_positive = raw_data[
    (raw_data["Quantity"] > 0) &
    (raw_data["UnitPrice"] == 0)
]

zero_price_positive.shape

(1179, 8)

In [12]:
zero_price_positive[
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "UnitPrice", "Description"]
].sort_values("Quantity", ascending=False).head(20)

,InvoiceNo,StockCode,Quantity,InvoiceDate,UnitPrice,Description
502122,578841,84826,12540,2011-11-25 15:57:00,0.0,ASSTD DESIGN 3D PAPER STICKERS
74614,542504,37413,5568,2011-01-28 12:03:00,0.0,NaN
220843,556231,85123A,4000,2011-06-09 15:04:00,0.0,?
263885,560040,23343,3100,2011-07-14 14:28:00,0.0,came coded as 20713
115807,546139,84988,3000,2011-03-09 16:35:00,0.0,?
74615,542505,79063D,2560,2011-01-28 12:04:00,0.0,NaN
160541,550460,47556B,1300,2011-04-18 13:18:00,0.0,did a credit and did not tick ret
203751,554550,47566B,1300,2011-05-25 09:57:00,0.0,incorrectly credited C550456 see 47
82795,543258,84611B,1287,2011-02-04 16:06:00,0.0,NaN
422750,573114,20713,1000,2011-10-27 15:36:00,0.0,wrongly coded 23343


In [13]:
zero_price_positive["Description"].isna().sum()

np.int64(592)

In [15]:
raw_data["ZeroPricePositiveQuantity"] = (
    (raw_data["Quantity"] > 0) &
    (raw_data["UnitPrice"] == 0)
)
raw_data["ZeroPricePositiveQuantity"].value_counts()

ZeroPricePositiveQuantity
False    540730
True       1179
Name: count, dtype: int64

In [16]:
zero_price_demand = raw_data.loc[
    raw_data["ZeroPricePositiveQuantity"],
    "Quantity"
].sum()

positive_demand = raw_data.loc[
    raw_data["Quantity"] > 0,
    "Quantity"
].sum()

zero_price_demand, positive_demand, zero_price_demand / positive_demand * 100

(np.int64(72603), np.int64(5660981), np.float64(1.2825162281943712))

In [20]:
zero_price_positive.sort_values(
    "Quantity",
    ascending=False
)[
    ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "Description"]
].head(20)

,InvoiceNo,StockCode,Quantity,InvoiceDate,Description
502122,578841,84826,12540,2011-11-25 15:57:00,ASSTD DESIGN 3D PAPER STICKERS
74614,542504,37413,5568,2011-01-28 12:03:00,NaN
220843,556231,85123A,4000,2011-06-09 15:04:00,?
263885,560040,23343,3100,2011-07-14 14:28:00,came coded as 20713
115807,546139,84988,3000,2011-03-09 16:35:00,?
74615,542505,79063D,2560,2011-01-28 12:04:00,NaN
160541,550460,47556B,1300,2011-04-18 13:18:00,did a credit and did not tick ret
203751,554550,47566B,1300,2011-05-25 09:57:00,incorrectly credited C550456 see 47
82795,543258,84611B,1287,2011-02-04 16:06:00,NaN
422750,573114,20713,1000,2011-10-27 15:36:00,wrongly coded 23343


In [30]:
# 1. Define valid, positive-price product transactions
good_mask = (
    raw_data["Quantity"].gt(0)
    & raw_data["UnitPrice"].gt(0)
)

# 2. Normalize descriptions for reliable matching
good_descriptions = set(
    raw_data.loc[good_mask, "Description"]
    .dropna()
    .str.strip()
    .str.casefold()
    .unique()
)

zero_descriptions = (
    zero_price_positive["Description"]
    .fillna("")
    .str.strip()
    .str.casefold()
)

# 3. Keep zero-price transactions whose descriptions
#    never occur in positive-quantity, positive-price sales
suspicious_zero_price = zero_price_positive.loc[
    ~zero_descriptions.isin(good_descriptions)
].copy()

suspicious_zero_price["Description"].unique().tolist()


[nan,
 'amazon',
 'amazon sales',
 '?',
 'Found',
 'found',
 'Adjustment',
 'wrongly sold (22719) barcode',
 'rcvd be air temp fix for dotcom sit',
 'did  a credit  and did not tick ret',
 'adjustment',
 'returned',
 'mailout ',
 'mailout',
 'on cargo order',
 'incorrectly credited C550456 see 47',
 'to push order througha s stock was ',
 'FOUND',
 'TRAVEL CARD WALLET DOTCOMGIFTSHOP',
 'came coded as 20713',
 'alan hodge cant mamage this section',
 'dotcom',
 'FBA',
 'test',
 'taig adjust',
 'allocate stock for dotcom orders ta',
 'add stock to allocate online orders',
 'for online retail orders',
 'Amazon',
 'found box',
 'damaged',
 'Found in w/hse',
 'website fixed',
 'Lighthouse Trading zero invc incorr',
 'michel oops',
 'wrongly coded 20713',
 'Had been put aside.',
 'Sale error',
 'Amazon Adjustment',
 'wrongly marked 23343',
 'Marked as 23343',
 'wrongly coded 23343',
 'Found by jackie',
 'check',
 'wrongly marked',
 'had been put aside',
 'amazon adjust',
 'dotcomstock',
 'Joh

In [38]:
suspicious_matches = raw_data.loc[
    raw_data["Quantity"].gt(0)
    & raw_data["Description"].isin(suspicious_good_price)
].copy()

display(suspicious_matches)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,ZeroPricePositiveQuantity


In [39]:
remaining_zero_price = zero_price_positive.loc[
    ~zero_price_positive["Description"].isin(
        suspicious_zero_price["Description"]
    )
].copy()

remaining_zero_price["Description"].unique().tolist()

['ROUND CAKE TIN VINTAGE GREEN',
 'CREAM SWEETHEART LETTER RACK',
 'ZINC WILLIE WINKIE  CANDLE STICK',
 'BOX OF 24 COCKTAIL PARASOLS',
 'DOORMAT ENGLISH ROSE ',
 'DOORMAT 3 SMILEY CATS',
 'GREEN REGENCY TEACUP AND SAUCER',
 'FRENCH BLUE METAL DOOR SIGN 7',
 'FRENCH BLUE METAL DOOR SIGN 5',
 'FRENCH BLUE METAL DOOR SIGN 6',
 'FRENCH BLUE METAL DOOR SIGN 4',
 'FRENCH BLUE METAL DOOR SIGN No',
 'FRENCH BLUE METAL DOOR SIGN 8',
 'FRENCH BLUE METAL DOOR SIGN 1',
 'RED KITCHEN SCALES',
 'IVORY KITCHEN SCALES',
 'SET OF 6 SOLDIER SKITTLES',
 'CHILDS GARDEN TROWEL BLUE ',
 'CHILDRENS GARDEN GLOVES BLUE',
 'PICNIC BASKET WICKER SMALL',
 'PICNIC BASKET WICKER LARGE',
 'EMPIRE UNION JACK TV DINNER TRAY',
 'TV DINNER TRAY VINTAGE PAISLEY',
 'SPACEBOY TV DINNER TRAY',
 'TV DINNER TRAY DOLLY GIRL',
 'CHILDS GARDEN SPADE BLUE',
 'CHILDS GARDEN RAKE BLUE',
 'WATERING CAN PINK BUNNY',
 'ENAMEL FIRE BUCKET CREAM',
 'ENAMEL FLOWER JUG CREAM',
 'AIRLINE BAG VINTAGE JET SET WHITE',
 'AIRLINE BAG VINTAGE JE

In [44]:
remaining_summary = (
    remaining_zero_price
    .groupby(["StockCode", "Description"], dropna=False)
    .agg(
        Transactions=("Quantity", "size"),
        TotalQuantity=("Quantity", "sum"),
        MaxQuantity=("Quantity", "max"),
    )
    .sort_values("TotalQuantity", ascending=False)
)

display(remaining_summary.head(30))

,,Transactions,TotalQuantity,MaxQuantity
StockCode,Description,,,
84826,ASSTD DESIGN 3D PAPER STICKERS,1,12540,12540
46000S,POLYESTER FILLER PAD 40x40cm,5,1070,320
46000M,POLYESTER FILLER PAD 45x45cm,5,680,240
23157,SET OF 6 NATIVITY MAGNETS,1,240,240
23234,BISCUIT TIN VINTAGE CHRISTMAS,1,216,216
22616,PACK OF 12 LONDON TISSUES,1,204,204
23268,SET OF 2 CERAMIC CHRISTMAS REINDEER,1,192,192
23298,SPOTTY BUNTING,2,171,170
21786,POLKADOT RAIN HAT,1,144,144


In [43]:
display(
    remaining_zero_price
    .sort_values("Quantity", ascending=False)
    [
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
        ]
    ]
    .head(30)
)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.0,13256.0
363657,568567,46000S,POLYESTER FILLER PAD 40x40cm,320,2011-09-27 18:41:00,0.0,NaN
538505,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN
538504,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN
298054,562973,23157,SET OF 6 NATIVITY MAGNETS,240,2011-08-11 11:42:00,0.0,14911.0
298302,563015,46000S,POLYESTER FILLER PAD 40x40cm,220,2011-08-11 12:24:00,0.0,NaN
436428,574138,23234,BISCUIT TIN VINTAGE CHRISTMAS,216,2011-11-03 11:26:00,0.0,12415.0
254131,559296,22616,PACK OF 12 LONDON TISSUES,204,2011-07-07 12:35:00,0.0,NaN
314746,564651,23268,SET OF 2 CERAMIC CHRISTMAS REINDEER,192,2011-08-26 14:19:00,0.0,14646.0
254132,559296,23298,SPOTTY BUNTING,170,2011-07-07 12:35:00,0.0,NaN


In [45]:
# Positive-price transactions for the same stock codes
normal_sales = raw_data.loc[
    raw_data["Quantity"].gt(0)
    & raw_data["UnitPrice"].gt(0)
].copy()

normal_summary = (
    normal_sales
    .groupby("StockCode")
    .agg(
        NormalTransactions=("Quantity", "size"),
        NormalMedianQuantity=("Quantity", "median"),
        NormalMaxQuantity=("Quantity", "max"),
    )
)

comparison = (
    remaining_zero_price
    .groupby("StockCode")
    .agg(
        ZeroPriceTransactions=("Quantity", "size"),
        ZeroPriceTotalQuantity=("Quantity", "sum"),
        ZeroPriceMaxQuantity=("Quantity", "max"),
    )
    .join(normal_summary, how="left")
    .sort_values("ZeroPriceMaxQuantity", ascending=False)
)

display(comparison.head(30))

,ZeroPriceTransactions,ZeroPriceTotalQuantity,ZeroPriceMaxQuantity,NormalTransactions,NormalMedianQuantity,NormalMaxQuantity
StockCode,,,,,,
84826,1,12540,12540,36,22.0,120
46000S,5,1070,320,185,4.0,125
23157,1,240,240,171,12.0,240
46000M,5,680,240,138,4.5,50
23234,1,216,216,280,6.0,216
22616,1,204,204,528,12.0,1296
23268,1,192,192,90,12.0,384
23298,2,171,170,1165,3.0,200
21786,1,144,144,234,2.0,144


## 7. Validate Cleaning Determinism

The production data-cleaning pipeline should be deterministic.

Running the same transformation on the same source data should produce the
same daily demand dataset every time.

In [8]:
from scripts.clean_data import classify_transactions, create_daily_sales

In [9]:
classified_1 = classify_transactions(raw_data)
daily_sales_1 = create_daily_sales(classified_1)

classified_2 = classify_transactions(raw_data)
daily_sales_2 = create_daily_sales(classified_2)

classified_1.equals(classified_2)

Classification completed in 0.36 seconds
Daily sales aggregation completed in 0.37 seconds
Classification completed in 0.31 seconds
Daily sales aggregation completed in 0.40 seconds


True

In [10]:
daily_sales_1 = daily_sales_1.sort_values(
    ["Date", "StockCode"]
).reset_index(drop=True)

daily_sales_2 = daily_sales_2.sort_values(
    ["Date", "StockCode"]
).reset_index(drop=True)

daily_sales_1.equals(daily_sales_2)

True

## Phase 1 — Data validation conclusion

The production cleaning pipeline passed all validation checks.

Product-level first and last dates were preserved when constructing the continuous forecasting series. No zero-demand observations were found outside each product's active history, and the forecasting dataset contains no missing values.

Total positive demand was preserved from the raw transaction data through daily aggregation and the forecasting series.

Exceptional transactions were investigated separately. Product `23843` contains an extreme quantity of 80,995 units followed by an exact cancellation 12 minutes later. Product `84611B` contains a large operational event involving a `thrown away` adjustment. These observations are retained for analysis rather than being removed automatically from the raw data.

The production cleaning functions were executed twice on the same raw input and produced identical outputs, confirming that the cleaning pipeline is deterministic.

The validated production pipeline therefore provides a reproducible daily product-level demand dataset for forecasting.